# 🛠️ CriminalID — Dataset Preparation Tool

**All-in-one notebook with 3 modes:**

| Mode | What it does | When to run |
|------|-------------|-------------|
| **1** | Convert grayscale dataset → RGB | Run **first**, once on existing dataset |
| **2** | Augment 200 → 1000+ images per person | Run after Mode 1, or after adding a new person |
| **3** | Capture new person via webcam | When adding someone new, then run Mode 2 again |

---
### ⚙️ Step 1: Update config below, then run cells top to bottom

In [16]:
# ─────────────────────────────────────────────────────────
#  GLOBAL CONFIG  ← Only change things here
# ─────────────────────────────────────────────────────────

DATASET_DIR = r"C:\Users\Lenovo\project\face_dataset"
TARGET_PER_PERSON = 1700   # Mode 2: target images per person
IMAGES_TO_CAPTURE = 200    # Mode 3: webcam captures per session
CAPTURE_DELAY_MS  = 150    # Mode 3: ms between captures (lower = faster)
IMG_SIZE          = (96, 96)    # matches MobileNetV2 training — 96×96 is faster

### 📦 Step 2: Import Libraries

In [17]:
import os
import cv2
import time
import random
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter
from pathlib import Path

random.seed(42)
np.random.seed(42)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


### 🔧 Step 3: Shared Utility Functions
*(Run this cell once — all modes below depend on it)*

In [18]:
def get_all_images(folder):
    """Get all jpg/jpeg/png images in a folder."""
    folder = Path(folder)
    return (list(folder.glob("*.jpg")) +
            list(folder.glob("*.jpeg")) +
            list(folder.glob("*.png")))

def get_person_folders(dataset_dir):
    """Get all subfolders (each = one person) in dataset."""
    return sorted([f for f in Path(dataset_dir).iterdir() if f.is_dir()])

def print_done(lines):
    print("\n" + "═"*60)
    for line in lines:
        print(f"  {line}")
    print("═"*60 + "\n")

print("✅ Utility functions ready")

✅ Utility functions ready


---
## 🟥 MODE 1 — Convert Grayscale Dataset → RGB

**Run this first** if your existing dataset was captured in grayscale.  
Converts every image to 3-channel RGB (in place). Images will still look grey but your model can read them correctly.  
Safe to run multiple times — already-RGB images are skipped automatically.

In [19]:
# ── MODE 1: Convert Grayscale → RGB ──────────────────────────

dataset_path = Path(DATASET_DIR)

if not dataset_path.exists():
    print(f"❌ Dataset folder not found: {DATASET_DIR}")
else:
    persons = get_person_folders(dataset_path)
    print(f"Found {len(persons)} person folders\n")
    print(f"{'Person':<42} {'Converted':>10} {'Skipped':>10}")
    print("-" * 65)

    total_converted = 0
    total_skipped   = 0
    total_failed    = 0

    for person_folder in persons:
        images    = get_all_images(person_folder)
        converted = 0

        for img_path in images:
            try:
                img = Image.open(img_path)
                if img.mode == "RGB":
                    total_skipped += 1
                    continue
                img.convert("RGB").save(img_path)
                converted       += 1
                total_converted += 1
            except Exception as e:
                print(f"  ⚠  Failed {img_path.name}: {e}")
                total_failed += 1

        skipped = len(images) - converted
        print(f"  ✓  {person_folder.name:<40} {converted:>10} {skipped:>10}")

    print_done([
        f"✓  Converted to RGB : {total_converted}",
        f"○  Already RGB      : {total_skipped}",
        f"✗  Failed           : {total_failed}",
        "",
        "✅  Dataset is now RGB!",
        "    Run MODE 2 below to augment images to 1000+ per person."
    ])

Found 13 person folders

Person                                      Converted    Skipped
-----------------------------------------------------------------
  ✓  Aniket_Kakad                                      0       1700
  ✓  Atharva_Ayachit                                   0       1700
  ✓  Chandrakant_Kshirsagar                            0       1700
  ✓  Dhananjay_Patil                                   0       1700
  ✓  Ishika_Mehre                                      0       1700
  ✓  Japneet_Singh                                     0       1700
  ✓  Jay_Kumar                                         0       1700
  ✓  Jyoti_Ayachit                                     0       1700
  ✓  Kajal_Yerone                                      0       1700
  ✓  Sakshi_Lahekar                                    0       1700
  ✓  Sayali_Chidrawar                                  0       1700
  ✓  Shreyas_Jadhav                                    0       1700
  ✓  Tushar_Patil           

---
## 🟧 MODE 2 — Augment Dataset to 1000+ Images Per Person

**Run after Mode 1**, or after adding a new person with Mode 3.  
For each original image it generates up to 12 variations:  
flip · rotate · brighter · darker · contrast · blur · sharpen · zoom · noise · flip+rotate · color jitter · shift  

200 originals × 12 = up to **2400 variations** — stops when `TARGET_PER_PERSON` is reached.

In [14]:
# ── Augmentation function (used inside Mode 2) ───────────────

def augment_image(img_pil, counter):
    """Returns list of (tag, PIL image) augmented variations."""
    augmented = []
    img_rgb   = img_pil.convert("RGB")
    img_arr   = np.array(img_rgb)
    w, h      = img_rgb.size

    flipped = img_rgb.transpose(Image.FLIP_LEFT_RIGHT)

    augmented.append(("flip",    flipped))
    augmented.append(("rot",     img_rgb.rotate(random.uniform(-20, 20), fillcolor=(128,128,128))))
    augmented.append(("bright",  ImageEnhance.Brightness(img_rgb).enhance(random.uniform(1.2, 1.6))))
    augmented.append(("dark",    ImageEnhance.Brightness(img_rgb).enhance(random.uniform(0.5, 0.8))))
    augmented.append(("contr",   ImageEnhance.Contrast(img_rgb).enhance(random.uniform(1.2, 1.8))))
    augmented.append(("blur",    img_rgb.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.8, 1.5)))))
    augmented.append(("sharp",   img_rgb.filter(ImageFilter.SHARPEN)))

    margin = int(min(w, h) * 0.10)
    augmented.append(("zoom",    img_rgb.crop((margin, margin, w-margin, h-margin)).resize((w, h), Image.LANCZOS)))

    noise = np.clip(img_arr.astype(np.float32) + np.random.normal(0, random.uniform(8,20), img_arr.shape), 0, 255).astype(np.uint8)
    augmented.append(("noise",   Image.fromarray(noise)))
    augmented.append(("fliprot", flipped.rotate(random.uniform(-15, 15), fillcolor=(128,128,128))))

    jitter = ImageEnhance.Color(img_rgb).enhance(random.uniform(0.7, 1.3))
    augmented.append(("jitter",  ImageEnhance.Brightness(jitter).enhance(random.uniform(0.85, 1.15))))

    sx, sy = random.randint(-20, 20), random.randint(-20, 20)
    augmented.append(("shift",   img_rgb.transform(img_rgb.size, Image.AFFINE, (1,0,sx,0,1,sy), fillcolor=(128,128,128))))

    return augmented


def augment_person_folder(person_folder, target):
    images   = get_all_images(person_folder)
    existing = len(images)

    # ── FIX: skip empty folders instead of crashing ──
    if existing == 0:
        print(f"  ⚠  SKIPPED (empty folder): {person_folder.name}")
        print(f"     → Add images to this folder then run Mode 2 again.")
        return 0

    needed = max(0, target - existing)
    if needed == 0:
        return 0

    generated = 0
    img_idx   = 0
    counter   = 0

    while generated < needed:
        source = images[img_idx % len(images)]
        img_idx += 1
        try:
            aug_list = augment_image(Image.open(source).convert("RGB"), counter)
            for tag, aug_img in aug_list:
                if generated >= needed:
                    break
                aug_img.save(person_folder / f"aug_{counter:04d}_{tag}.jpg", quality=92)
                generated += 1
                counter   += 1
        except Exception as e:
            print(f"    ⚠  Skipping {source.name}: {e}")

    return generated

print("✅ Augmentation functions ready")

✅ Augmentation functions ready


In [15]:
# ── MODE 2: Run Augmentation ──────────────────────────────────

dataset_path = Path(DATASET_DIR)

if not dataset_path.exists():
    print(f"❌ Dataset folder not found: {DATASET_DIR}")
else:
    persons = get_person_folders(dataset_path)
    print(f"Found {len(persons)} person folders")
    print(f"Target per person: {TARGET_PER_PERSON} images\n")

    # ── Pre-scan: catch empty folders before starting ──
    empty_folders = [p for p in persons if len(get_all_images(p)) == 0]
    if empty_folders:
        print("⚠  WARNING — These folders are EMPTY and will be skipped:")
        for ef in empty_folders:
            print(f"   → {ef.name}  (add images here then rerun Mode 2)")
        print()

    ready_folders = [p for p in persons if len(get_all_images(p)) > 0]
    print(f"{'Person':<37} {'Before':>7} {'After':>7}  Progress")
    print("-" * 70)

    total_new = 0

    for person_folder in ready_folders:
        before   = len(get_all_images(person_folder))
        new      = augment_person_folder(person_folder, TARGET_PER_PERSON)
        after    = before + new
        total_new += new

        filled = min(30, int(after / TARGET_PER_PERSON * 30))
        bar    = "█" * filled + "░" * (30 - filled)
        pct    = min(100, int(after / TARGET_PER_PERSON * 100))
        print(f"  {person_folder.name:<35} {before:>7} {after:>7}  [{bar}] {pct}%")

    print_done([
        f"✓  Total new images generated : {total_new}",
        f"✓  Folders processed          : {len(ready_folders)}",
        f"⚠  Folders skipped (empty)    : {len(empty_folders)}",
        "",
        "✅  Now retrain your model in Criminal_Face_Recognition.ipynb!"
    ])

Found 13 person folders
Target per person: 1700 images

Person                                 Before   After  Progress
----------------------------------------------------------------------
  Aniket_Kakad                           1700    1700  [██████████████████████████████] 100%
  Atharva_Ayachit                        1700    1700  [██████████████████████████████] 100%
  Chandrakant_Kshirsagar                 1700    1700  [██████████████████████████████] 100%
  Dhananjay_Patil                        1700    1700  [██████████████████████████████] 100%
  Ishika_Mehre                           1700    1700  [██████████████████████████████] 100%
  Japneet_Singh                          1700    1700  [██████████████████████████████] 100%
  Jay_Kumar                              1700    1700  [██████████████████████████████] 100%
  Kajal_Yerone                           1700    1700  [██████████████████████████████] 100%
  New_Atharva                             200    1700  [█████████

---
## 🟩 MODE 3 — Capture New Person via Webcam

Use this to **add a brand new person** to your dataset.  
Set `PERSON_NAME` below, then run the cell.  
A webcam window opens — press **SPACE** to start capturing, **Q** to stop.

After this, go back and **run Mode 2** to augment the new person to 1000+ images,  
then update `criminals_info.csv` and `class_names.txt`, and retrain.

In [7]:
# ── MODE 3 CONFIG ← Change this before running ───────────────
PERSON_NAME = "New_person"   # ← use underscores e.g. John_Doe

In [9]:
# ── MODE 3: Webcam Capture ────────────────────────────────────
# Uses 3 cascades: front face, alt front, left & right profile.
# Detects angled/tilted faces — move your head freely during capture.

save_dir  = Path(DATASET_DIR) / PERSON_NAME
save_dir.mkdir(parents=True, exist_ok=True)

existing  = get_all_images(save_dir)
start_idx = len(existing)

print(f"  📁 Saving to   : {save_dir}")
print(f"  📸 Will capture: {IMAGES_TO_CAPTURE} images")
print(f"  📂 Existing    : {start_idx} (new images continue from here)")
print(f"\n  Press  SPACE  → START / PAUSE")
print(f"  Press  Q      → Quit (saves progress)")
print(f"  💡 TIP: Slowly move your head left, right, up, down for better dataset\n")

# ── Load 3 cascades: front, alt-front, profile ────────────────
cascade_front   = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
cascade_alt     = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')
cascade_profile = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_profileface.xml')

def detect_any_face(gray_frame):
    """
    Tries 4 detection methods in order until one finds a face:
      1. Front face (loose settings)
      2. Alt front face (better for tilted faces)
      3. Left profile
      4. Right profile (flipped frame trick)
    Returns (faces_array, method_name)
    """
    # 1. Standard frontal — loose minNeighbors + small minSize catches tilts
    faces = cascade_front.detectMultiScale(
        gray_frame, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
    )
    if len(faces) > 0:
        return faces, 'frontal'

    # 2. Alt frontal cascade — handles up to ~30 degree head tilt
    faces = cascade_alt.detectMultiScale(
        gray_frame, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
    )
    if len(faces) > 0:
        return faces, 'alt-frontal'

    # 3. Left profile
    faces = cascade_profile.detectMultiScale(
        gray_frame, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
    )
    if len(faces) > 0:
        return faces, 'profile-left'

    # 4. Right profile — flip frame, detect left profile, mirror coords back
    flipped = cv2.flip(gray_frame, 1)
    faces   = cascade_profile.detectMultiScale(
        flipped, scaleFactor=1.05, minNeighbors=3, minSize=(50, 50)
    )
    if len(faces) > 0:
        fw    = gray_frame.shape[1]
        faces = np.array([(fw - x - w, y, w, h) for (x, y, w, h) in faces])
        return faces, 'profile-right'

    return [], 'none'


cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("❌ Could not open webcam. Check that it is connected.")
else:
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    count     = 0
    capturing = False
    last_time = time.time()
    last_mode = 'none'

    while True:
        ret, frame = cap.read()
        if not ret:
            print("❌ Failed to read from webcam.")
            break

        gray              = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces, last_mode  = detect_any_face(gray)

        display = frame.copy()

        # ── Top status bar ────────────────────────────────────────
        status_color = (0, 200, 0) if capturing else (0, 130, 255)
        status_text  = f"CAPTURING {count}/{IMAGES_TO_CAPTURE}" if capturing else "PAUSED — Press SPACE to start"
        cv2.rectangle(display, (0, 0), (640, 40), (20, 20, 20), -1)
        cv2.putText(display, status_text, (10, 27),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, status_color, 2)

        # ── Bottom label: person name + which cascade found the face ──
        cv2.putText(display, f"Person: {PERSON_NAME}   Detected via: {last_mode}",
                    (10, 470), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180, 180, 180), 1)

        if len(faces) > 0:
            # Use largest detected face
            x, y, w, h = sorted(faces, key=lambda f: f[2] * f[3], reverse=True)[0]

            # Clamp coords so crop never goes out of frame boundaries
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(frame.shape[1], x + w)
            y2 = min(frame.shape[0], y + h)

            box_color = (0, 255, 0) if capturing else (100, 100, 255)
            cv2.rectangle(display, (x1, y1), (x2, y2), box_color, 2)

            if capturing and (time.time() - last_time) > CAPTURE_DELAY_MS / 1000.0:
                face_crop = frame[y1:y2, x1:x2]

                if face_crop.size > 0:   # safety: skip empty crops
                    face_rgb  = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
                    img_pil   = Image.fromarray(face_rgb).resize(IMG_SIZE)
                    save_path = save_dir / f"img_{start_idx + count:04d}.jpg"
                    img_pil.save(str(save_path), quality=92)

                    count    += 1
                    last_time = time.time()
                    # Flash white box on save
                    cv2.rectangle(display, (x1, y1), (x2, y2), (255, 255, 255), 3)

                    if count >= IMAGES_TO_CAPTURE:
                        break
        else:
            if capturing:
                cv2.putText(display, "NO FACE — try adjusting angle or lighting",
                            (80, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 60, 255), 2)

        # ── Progress bar ──────────────────────────────────────────
        progress = int((count / IMAGES_TO_CAPTURE) * 620)
        cv2.rectangle(display, (10, 455), (630, 465), (40, 40, 40), -1)
        cv2.rectangle(display, (10, 455), (10 + progress, 465), (0, 200, 0), -1)

        cv2.imshow(f"CriminalID — Capturing: {PERSON_NAME}", display)

        key = cv2.waitKey(1) & 0xFF
        if key == ord(' '):
            capturing = not capturing
            print(f"  {'▶  Capturing...' if capturing else '⏸  Paused.'}")
        elif key == ord('q') or count >= IMAGES_TO_CAPTURE:
            break

    cap.release()
    cv2.destroyAllWindows()


  📁 Saving to   : C:\Users\Lenovo\project\face_dataset\New_Atharva
  📸 Will capture: 200 images
  📂 Existing    : 0 (new images continue from here)

  Press  SPACE  → START / PAUSE
  Press  Q      → Quit (saves progress)
  💡 TIP: Slowly move your head left, right, up, down for better dataset

  ▶  Capturing...
